<a href="https://colab.research.google.com/github/craytonfu-max/ValorantAIAnalyzer/blob/main/notebooks/loading_screen_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Valorant AI Companion — Comp Classifier Pipeline

Loading screen → structured comp data (map + agent splash art classification).

**Setup:** run all cells top to bottom. Requires Google Drive mounted for persistent storage.

**Sections:**
1. Setup (imports, Drive mount)
2. Agent splash art cropping
3. Dataset building
4. Training
5. Map Name OCR


### Setup

In [ ]:
# import drive, shutil and os
from PIL import Image
import os
import shutil
from google.colab import drive

# Map OCR
!apt-get install -y tesseract-ocr
!pip install pytesseract
!pip install -U sympy

# Training Loop
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

# Map OCR
import pytesseract
from PIL import ImageOps
import numpy as np
from difflib import get_close_matches

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [ ]:
# allow drive access
drive.mount('/content/drive')

Mounted at /content/drive


### Archived Workflow: Manual File Addition, Cropping, and Moving

In [ ]:
# upload screenshots to temporary memory
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
# create raw_screenshots and crops folder in drive
os.makedirs('/content/drive/MyDrive/ValorantAI/raw_screenshots', exist_ok=True)
os.makedirs('/content/drive/MyDrive/ValorantAI/crops', exist_ok=True)

In [ ]:
# TODO: add 88 screenshots directly to drive storage
# move screenshots from temporary memory to drive storage
shutil.move("VALORANT-Win64-Shipping_HXiFbGmYNm.jpg", "/content/drive/MyDrive/ValorantAI/raw_screenshots/VALORANT-Win64-Shipping_HXiFbGmYNm.jpg")
shutil.move("VALORANT-Win64-Shipping_TULMrdf0EL.jpg", "/content/drive/MyDrive/ValorantAI/raw_screenshots/VALORANT-Win64-Shipping_TULMrdf0EL.jpg")
shutil.move("VALORANT-Win64-Shipping_kdt1NqB5tJ.jpg", "/content/drive/MyDrive/ValorantAI/raw_screenshots/VALORANT-Win64-Shipping_kdt1NqB5tJ.jpg")

### Agent Splash Art Cropping (base crop function and region intialization)

In [ ]:
# takes an image and saves important regions by cropping and prints message indicating the base image has been cropped into x amount of pieces
regions_pct = {
    "ally_1": (55/1920, 180/1080, 170/1920, 290/1080),
    "ally_2": (55/1920, 350/1080, 170/1920, 459/1080),
    "ally_3": (55/1920, 515/1080, 170/1920, 626/1080),
    "ally_4": (55/1920, 680/1080, 170/1920, 792/1080),
    "ally_5": (55/1920, 847/1080, 170/1920, 959/1080),
    "enemy_1": (1750/1920, 180/1080, 1868/1920, 292/1080),
    "enemy_2": (1750/1920, 350/1080, 1868/1920, 459/1080),
    "enemy_3": (1750/1920, 515/1080, 1868/1920, 626/1080),
    "enemy_4": (1750/1920, 680/1080, 1868/1920, 792/1080),
    "enemy_5": (1750/1920, 847/1080, 1868/1920, 959/1080),
}

def crop_region_by_percent(img, left_pct, top_pct, right_pct, bottom_pct):
    width, height = img.size
    box = (
        int(left_pct * width),
        int(top_pct * height),
        int(right_pct * width),
        int(bottom_pct * height),
    )
    return img.crop(box)

def crop_splash_art(image_path, output_dir):
    img = Image.open(image_path)
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    os.makedirs(output_dir, exist_ok=True)

    for name, (l, t, r, b) in regions_pct.items():
        crop = crop_region_by_percent(img, l, t, r, b)
        crop.save(f"{output_dir}/{base_name}_{name}.png")

    print(f"Cropped {base_name} into {len(regions_pct)} pieces")


def crop_all_screenshots(input_dir, output_dir):
    count = 0
    skipped = 0
    for filename in os.listdir(input_dir):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            base_name = os.path.splitext(filename)[0]
            check_file = os.path.join(output_dir, f"{base_name}_ally_1.png")  # changed from _map.png
            if os.path.exists(check_file):
                skipped += 1
                continue
            full_path = os.path.join(input_dir, filename)
            crop_splash_art(full_path, output_dir)
            count += 1
    print(f"Processed {count} new screenshots, skipped {skipped} already done.")

    # debug for map
    # print(regions["map"])

### Manual Cropping

In [ ]:
# crop the three base images
crop_splash_art(
    "/content/drive/MyDrive/ValorantAI/raw_screenshots/VALORANT-Win64-Shipping_HXiFbGmYNm.jpg",
    "/content/drive/MyDrive/ValorantAI/crops"
)

crop_splash_art(
    "/content/drive/MyDrive/ValorantAI/raw_screenshots/VALORANT-Win64-Shipping_TULMrdf0EL.jpg",
    "/content/drive/MyDrive/ValorantAI/crops"
)

crop_splash_art(
    "/content/drive/MyDrive/ValorantAI/raw_screenshots/VALORANT-Win64-Shipping_kdt1NqB5tJ.jpg",
    "/content/drive/MyDrive/ValorantAI/crops"
)

Cropped VALORANT-Win64-Shipping_HXiFbGmYNm into 11 pieces
Cropped VALORANT-Win64-Shipping_TULMrdf0EL into 11 pieces
Cropped VALORANT-Win64-Shipping_kdt1NqB5tJ into 11 pieces


### Grid Debugging

In [ ]:
from PIL import Image, ImageDraw

# debug grid to find cropping coordinates
def show_grid(image_path, output_path="grid_debug.png", step=20):
    img = Image.open(image_path).resize((1920, 1080)).convert("RGB")
    draw = ImageDraw.Draw(img)

    # horizontal lines (y-axis labels, reading top to bottom)
    for y in range(0, 1080, step):
        draw.line([(0, y), (1920, y)], fill=(255, 0, 0), width=1)
        draw.text((5, y), str(y), fill=(255, 255, 0))

    # vertical lines (x-axis labels, reading left to right)
    for x in range(0, 1920, step):
        draw.line([(x, 0), (x, 1080)], fill=(0, 200, 255), width=1)
        draw.text((x, 5), str(x), fill=(255, 255, 0))

    img.save(output_path)
    print(f"Saved grid overlay to {output_path}")

show_grid("/content/drive/MyDrive/ValorantAI/raw_screenshots/VALORANT-Win64-Shipping_HXiFbGmYNm.jpg")

Saved grid overlay to grid_debug.png


### Batch Crop Function

In [ ]:
# Batch Version of cropping pipeline for all images in raw_screenshots that have not been previously cropped by checking if the map.png exists for the raw screenshot
def crop_all_screenshots(input_dir, output_dir):
    count = 0
    skipped = 0
    for filename in os.listdir(input_dir):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            base_name = os.path.splitext(filename)[0]
            check_file = os.path.join(output_dir, f"{base_name}_map.png")
            if os.path.exists(check_file):
                skipped += 1
                continue  # already cropped, skip it
            full_path = os.path.join(input_dir, filename)
            crop_splash_art(full_path, output_dir)
            count += 1
    print(f"Processed {count} new screenshots, skipped {skipped} already done.")

crop_all_screenshots(
    "/content/drive/MyDrive/ValorantAI/raw_screenshots",
    "/content/drive/MyDrive/ValorantAI/crops"
)

Cropped VALORANT-Win64-Shipping_ux7E3f3nYc into 11 pieces
Cropped VALORANT-Win64-Shipping_WfognoCJQY into 11 pieces
Cropped VALORANT-Win64-Shipping_rsrzAZTtxQ into 11 pieces
Processed 3 new screenshots, skipped 31 already done.


### Database Initialization

In [ ]:
import pandas as pd

# dataset building
df = pd.read_csv("/content/drive/MyDrive/ValorantAI/labels.csv")                # load the google sheet downloaded as labels.csv into a dataframe

crops_dir = "/content/drive/MyDrive/ValorantAI/crops"                           # shortcut variables to reduce repetition
training_dir = "/content/drive/MyDrive/ValorantAI/training_data"

slot_columns = ["ally_1", "ally_2", "ally_3", "ally_4", "ally_5",
                 "enemy_1", "enemy_2", "enemy_3", "enemy_4", "enemy_5"]

copied = 0
skipped = 0

for _, row in df.iterrows():
    base_name = os.path.splitext(row["filename"])[0]                            # get base file names like VALORANT-Win64-Shipping_HXiFbGmYNm
    for slot in slot_columns:
        agent_name = row[slot]
        if pd.isna(agent_name):
            continue                                                            # if column blank, continue

        safe_agent_name = agent_name.replace("/", "-")                          # sanitize agent name for folder use - "/" is a path separator,
                                                                                # so "KAY/O" would otherwise create a KAY folder with an O subfolder
                                                                                # instead of one folder named "KAY/O"

        crop_filename = f"{base_name}_{slot}.png"
        crop_path = os.path.join(crops_dir, crop_filename)                      # build full crop file name
        if not os.path.exists(crop_path):
            continue                                                            # skip if crop is missing
        class_dir = os.path.join(training_dir, safe_agent_name)                 # from, take from training data (uses sanitized name)
        dest_path = os.path.join(class_dir, crop_filename)                      # destination, send to agent folder in training data

        if os.path.exists(dest_path):
            skipped += 1
            continue                                                            # already copied, skip it

        os.makedirs(class_dir, exist_ok=True)                                   # make agent folder in training directory, ok if exists already
        shutil.copy(crop_path, dest_path)                                       # copy crop image into folder from crops folder to agent training data
        copied += 1

print(f"Copied {copied} new images, skipped {skipped} already done.")

Copied 3 new images, skipped 337 already done.


### AI Training Loop

In [ ]:
# debug to see if we are using CPU or GPU (True if using GPU)
print(torch.cuda.is_available())

True


In [ ]:
# Transforms — resize + normalize our data to match what the pretrained model expects
transform = transforms.Compose([
    transforms.Resize((224, 224)),                                                # resizes every image to 224x224 pizels since ResNet expects fixed input size
    transforms.ToTensor(),                                                        # converts image into PyTorch tensor (Multi-dimensional array) that the model can operate on
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # shifts and scales tensor values based on ResNets original dataset helping the model "see" better
])

In [ ]:
dataset = datasets.ImageFolder("/content/drive/MyDrive/ValorantAI/training_data", transform=transform)
print(f"Classes found: {dataset.classes}")
print(f"Total images: {len(dataset)}")

Classes found: ['Astra', 'Breach', 'Brimstone', 'Chamber', 'Clove', 'Cypher', 'Deadlock', 'Fade', 'Gekko', 'Harbor', 'Iso', 'Jett', 'KAY', 'Killjoy', 'Miks', 'Neon', 'Omen', 'Phoenix', 'Raze', 'Reyna', 'Sage', 'Skye', 'Sova', 'Tejo', 'Veto', 'Viper', 'Vyse', 'Waylay', 'Yoru']
Total images: 310


In [ ]:
# Load dataset from our folder structure
dataset = datasets.ImageFolder("/content/drive/MyDrive/ValorantAI/training_data", transform=transform)
print(f"Classes found: {dataset.classes}")
print(f"Total images: {len(dataset)}")

# Split into train/test with an 80/20 split (80% for training, 20% for testing)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_data, test_data = random_split(dataset, [train_size, test_size])  # random_split picks random image into each group

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)       # DataLoader feeds images in batches, shuffle true is creating a random order
test_loader = DataLoader(test_data, batch_size=8, shuffle=False)

# Load pretrained model (ResNet-18), swap final layer for our classes
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # checks for GPU otherwise falls back to CPU
model = model.to(device)                                                # moves model's data to whichever device was chosen
print(f"Training on: {device}")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()                                       # define loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)              # initialize optimizer Adam with step size 0.001

# Training loop
epochs = 5                                                              # epoch is one pass through the training loop
for epoch in range(epochs):
    model.train()                                                       # set model to training mode
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)           # allow gpu to access images and labels
        optimizer.zero_grad()                                           # zero gradients
        outputs = model(images)                                         # make predictions
        loss = criterion(outputs, labels)                               # calculate how incorrect model was
        loss.backward()                                                 # update gradients (how model should change weights)
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

# Evaluate
model.eval()                                                            # set model to evaluation mode

correct = 0                                                             # counter: how many predictions were right
total = 0                                                               # counter: how many test images we checked overall

with torch.no_grad():                                                   # disable gradient tracking - no training, just checking accuracy

    for images, labels in test_loader:                                  # loop through the test set, one batch at a time
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)                                         # forward pass: run the images through the model, get raw prediction scores
                                                                          # per class for every image in the batch

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)                                         # add this batch's image count to the running total
        correct += (predicted == labels).sum().item()                   # compare guesses to the true labels - True/False per image, sum them up
                                                                          # to count how many were correct in this batch, add to running total

print(f"Test accuracy: {100 * correct / total:.2f}% ({correct}/{total})")

# Save the trained model to Drive so it persists after the session ends
torch.save(model.state_dict(), "/content/drive/MyDrive/ValorantAI/agent_classifier.pth")
print("Model saved.")

Classes found: ['Astra', 'Breach', 'Brimstone', 'Chamber', 'Clove', 'Cypher', 'Deadlock', 'Fade', 'Gekko', 'Harbor', 'Iso', 'Jett', 'KAY-O', 'Killjoy', 'Miks', 'Neon', 'Omen', 'Phoenix', 'Raze', 'Reyna', 'Sage', 'Skye', 'Sova', 'Tejo', 'Veto', 'Viper', 'Vyse', 'Waylay', 'Yoru']
Total images: 340
Training on: cuda
Epoch 1/5 - Loss: 1.2894
Epoch 2/5 - Loss: 0.3765
Epoch 3/5 - Loss: 0.2029
Epoch 4/5 - Loss: 0.1184
Epoch 5/5 - Loss: 0.0983
Test accuracy: 100.00% (68/68)
Model saved.


### Map Name OCR

In [ ]:
# list of valid maps in valorant
valid_maps = ["Ascent", "Bind", "Breeze", "Fracture", "Haven", "Icebox",
              "Lotus", "Pearl", "Split", "Sunset", "Abyss", "Corrode", "Summit"]

# fuzzy-matches a garbled OCR string to the closest real map name,
# so small OCR noise (extra characters, minor misreads) doesn't
# automatically fail to match a valid map
def closest_map(ocr_text):
    matches = get_close_matches(ocr_text.upper(), [m.upper() for m in valid_maps], n=1, cutoff=0.3)         # n=1 means best match, cutoff being at least 30% similar,
                                                                                                            # cast both to uppercase to avoid case differences
    if matches:
        idx = [m.upper() for m in valid_maps].index(matches[0])                                             # get index of best match
        return valid_maps[idx]
    return "Unknown"

In [ ]:
#
def read_map_name(image_path):
    img = Image.open(image_path)
    width, height = img.size

    left = int((680/1920) * width)
    top = int((139/1080) * height)
    right = int((1260/1920) * width)
    bottom = int((251/1080) * height)

    crop = img.crop((left, top, right, bottom))
    crop_gray = crop.convert("L")
    crop_upscaled = crop_gray.resize((crop_gray.width * 3, crop_gray.height * 3), Image.LANCZOS)
    crop_upscaled = ImageOps.autocontrast(crop_upscaled)

    arr = np.array(crop_upscaled)
    binary = (arr > 128) * 255
    binary_img = Image.fromarray(binary.astype('uint8'))

    raw_text_7 = pytesseract.image_to_string(binary_img, config='--psm 7').strip()
    raw_text_8 = pytesseract.image_to_string(binary_img, config='--psm 8').strip()

    # psm 7 is only trustworthy for these specific maps
    psm7_reliable_maps = {"Breeze", "Icebox", "Lotus"}

    matched_7 = closest_map(raw_text_7)
    if matched_7 in psm7_reliable_maps:
        return matched_7, raw_text_7

    matched_8 = closest_map(raw_text_8)
    return matched_8, raw_text_8

In [ ]:
ocr_to_map = {
    "BREEZE": "Breeze",
    "IGEBOX": "Icebox",
    "SH TT": "Summit",
    "UAT": "Split",
    "SHAY": "Sunset",
    "LOTUS": "Lotus",
    "Ua]": "Pearl",
}

def map_from_raw(raw_text):
    return ocr_to_map.get(raw_text.strip(), "Unknown")

In [ ]:
def read_all_maps(input_dir):
    results = {}
    for filename in os.listdir(input_dir):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            full_path = os.path.join(input_dir, filename)
            matched_map, raw_text = read_map_name(full_path)
            results[filename] = (matched_map, raw_text)
            print(f"{filename}: matched='{matched_map}', raw='{raw_text}'")
    return results

map_results = read_all_maps("/content/drive/MyDrive/ValorantAI/raw_screenshots")

VALORANT-Win64-Shipping_HXiFbGmYNm.jpg: matched='Breeze', raw='BREEZE'
VALORANT-Win64-Shipping_kdt1NqB5tJ.jpg: matched='Summit', raw='SUMMIT'
VALORANT-Win64-Shipping_TULMrdf0EL.jpg: matched='Icebox', raw='IGEBOX'
VALORANT-Win64-Shipping_ezy4Vy8BdS.jpg: matched='Bind', raw='BIND |'
VALORANT-Win64-Shipping_Zm3qCoRBwL.jpg: matched='Split', raw='SPLIT'
VALORANT-Win64-Shipping_dSQcF2Y8vg.jpg: matched='Sunset', raw='SUNSET'
VALORANT-Win64-Shipping_L7A0P8sGbS.jpg: matched='Split', raw='SPLIT'
VALORANT-Win64-Shipping_ZweIENXHDc.jpg: matched='Summit', raw='SUMMIT'
VALORANT-Win64-Shipping_hPiaNSFBqo.jpg: matched='Breeze', raw='BREEZE'
VALORANT-Win64-Shipping_HgMiGJbOgZ.jpg: matched='Lotus', raw='LOTUS'
VALORANT-Win64-Shipping_xZxYwOvNin.jpg: matched='Haven', raw='HAVEN |'
VALORANT-Win64-Shipping_qL2XpFugXQ.jpg: matched='Pearl', raw='PEARL'
VALORANT-Win64-Shipping_uKFpKI3Oxb.jpg: matched='Sunset', raw='SUNSET'
VALORANT-Win64-Shipping_YL6espe3Aq.jpg: matched='Lotus', raw='LOTUS'
VALORANT-Win64-Shi

### Wrapper

In [ ]:
import json
with open("/content/drive/MyDrive/ValorantAI/class_names.json", "w") as f:
    json.dump(dataset.classes, f)

In [ ]:
with open("/content/drive/MyDrive/ValorantAI/class_names.json") as f:
    class_names = json.load(f)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
import json
from PIL import Image

# Load class names and model once, reuse across calls
with open("/content/drive/MyDrive/ValorantAI/class_names.json") as f:
    class_names = json.load(f)

num_classes = len(class_names)
inference_model = models.resnet18(weights=None)
inference_model.fc = nn.Linear(inference_model.fc.in_features, num_classes)
inference_model.load_state_dict(torch.load("/content/drive/MyDrive/ValorantAI/agent_classifier.pth"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model = inference_model.to(device)
inference_model.eval()  # inference mode, not training

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Override KAY/O naming as / is the escape character
display_name_overrides = {
    "KAY-O": "KAY/O",
}

# Takes a single PIL image (one agent splash art crop), returns predicted agent name.
def predict_agent(crop_image):
    img_tensor = inference_transform(crop_image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = inference_model(img_tensor)
        _, predicted_idx = torch.max(output, 1)

    predicted_class = class_names[predicted_idx.item()]
    return display_name_overrides.get(predicted_class, predicted_class)

# Takes a full screenshot, returns predicted ally and enemy team comps.
def classify_agents(image_path):
    img = Image.open(image_path)

    ally_team = []
    enemy_team = []

    for name, (l, t, r, b) in regions_pct.items():
        crop = crop_region_by_percent(img, l, t, r, b)
        agent = predict_agent(crop)
        if name.startswith("ally"):
            ally_team.append(agent)
        else:
            enemy_team.append(agent)

    return ally_team, enemy_team

In [ ]:
import inspect
print(inspect.getsource(predict_agent))

def predict_agent(crop_image):
    """Takes a single PIL image (one agent splash art crop), returns predicted agent name."""
    img_tensor = inference_transform(crop_image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = inference_model(img_tensor)
        _, predicted_idx = torch.max(output, 1)

    predicted_class = class_names[predicted_idx.item()]
    return display_name_overrides.get(predicted_class, predicted_class)



In [ ]:
def analyze_screenshot(image_path):
    map_name, _ = read_map_name(image_path)
    ally_team, enemy_team = classify_agents(image_path)
    return {
        "map": map_name,
        "ally_team": ally_team,
        "enemy_team": enemy_team
    }

In [ ]:
def analyze_screenshot(image_path):
    map_name, _ = read_map_name(image_path)
    ally_team, enemy_team = classify_agents(image_path)  # you'd build this wrapper around your trained model
    return {
        "map": map_name,
        "ally_team": ally_team,
        "enemy_team": enemy_team
    }

### Testing

In [ ]:
def test_all_screenshots(input_dir):
    for filename in os.listdir(input_dir):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            full_path = os.path.join(input_dir, filename)
            result = analyze_screenshot(full_path)
            print(f"{filename}:")
            print(f"  Map: {result['map']}")
            print(f"  Ally: {result['ally_team']}")
            print(f"  Enemy: {result['enemy_team']}")
            print()

test_all_screenshots("/content/drive/MyDrive/ValorantAI/test_screenshots")

VALORANT-Win64-Shipping_f55xYAykmL.jpg:
  Map: Split
  Ally: ['Fade', 'Chamber', 'Omen', 'Raze', 'Phoenix']
  Enemy: ['Phoenix', 'Neon', 'Omen', 'Sage', 'KAY/O']

VALORANT-Win64-Shipping_VLfz0nUpTP.jpg:
  Map: Haven
  Ally: ['Sova', 'Sage', 'Jett', 'Reyna', 'Clove']
  Enemy: ['Reyna', 'Jett', 'Skye', 'Clove', 'Chamber']

